# 📋 [OPS08] 드론 경보 시 관측자-지휘자-경계조 역할분담 모의훈련 (학생 실습용 워크북)
### AI Co-pilot 협업 기반 소부대 대공 방어 표준운용절차(SOP) 및 모의훈련 워크북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/OPS08/OPS08_workbook.ipynb)

---

## 📌 실습 개요 및 AI Co-pilot 활용 지침
- **도구 환경:** Python 3.10+, Pandas, Google Colab / Jupyter
- **AI 협업 원칙 (HITL):** AI는 체크리스트 초안, 시나리오 확장, AAR 4M 원인 분석의 보조 도구로 활용하며, 교전수칙 준수 및 안전 결심은 대원이 최종 확정합니다.
- **군사 보안 주의:** 모든 실습에는 제공된 가상 데이터셋만 사용하며 실제 군사 비밀은 일절 입력하지 않습니다.
---

## 💡 실습 환경별 산출물(docs/, sop/, logs/ 등) 편철 안내 (A안 & B안)
- **방법 A. 노트북 내 자동 파일 추출 (추천):** 코드 셀 첫 줄에 `%%writefile docs/ops08_baseline_assessment.md`와 같은 매직 커맨드를 선언하고 실행하면 가상 런타임에 폴더와 마크다운 파일이 자동 생성·저장됩니다. (AI Co-pilot에게 *'이 산출물을 %%writefile을 사용해 해당 경로 파일로 저장해줘'*라고 질의하면 편리합니다.)
- **방법 B. 단일 노트북 통합 제출 인정 (초보자 안심 대안):** 별도 마크다운 파일로 일일이 분리하지 않더라도, 워크북의 각 미션별 셀에 산출물 내용을 작성하고 실행 로그가 보존된 `OPS08_student.ipynb` 1개만 제출해도 동일하게 정상 완료로 인정됩니다.
- **로컬 IDE 환경 (VS Code, Cursor 등):** 프로젝트 작업 폴더 내에 `docs/`, `sop/`, `scenarios/`, `logs/`, `package/` 디렉토리를 생성하고 마크다운 파일을 직접 저장하여 관리합니다.


In [ ]:
# [OPS08] 소부대 드론 경보 모의훈련 자동 분석 및 검증 헬퍼 모듈
import pandas as pd
import numpy as np

def verify_mission1_raci(raci_data, checklist_items):
    """미션 1 RACI 매트릭스 및 체크리스트 무결성 검증"""
    print("=" * 60)
    print("📋 [미션 1 검증] 7단계 역할분담 매트릭스 & 체크리스트 검증")
    print("=" * 60)
    
    stages = ["포착", "식별", "보고", "전파", "결심", "대피·대응", "해제"]
    stage_pass = True
    for s in stages:
        if s not in raci_data:
            print(f"❌ 누락된 단계: {s}")
            stage_pass = False
        else:
            roles = raci_data[s]
            acc_count = sum(1 for r, duty in roles.items() if "책" in duty or "A" in duty)
            if acc_count != 1:
                print(f"⚠️ [{s} 단계] 책임자(A/책)가 정확히 1명이 아닙니다! (현재 {acc_count}명)")
                stage_pass = False
    
    check_count = len(checklist_items)
    check_pass = check_count >= 15
    print(f"✔ 현장 체크리스트 항목 수: {check_count}개 / 기준 15개 이상 {'[PASS]' if check_pass else '[FAIL]'}")
    print(f"✔ 7단계 RACI 매트릭스 책임자 무결성: {'[PASS]' if stage_pass else '[FAIL]'}")
    
    if stage_pass and check_pass:
        print("\n🎉 [미션 1 PASS] 현장 점검 기준과 역할 분담이 전술 기준에 완벽히 부합합니다!")
    else:
        print("\n❌ [보완 필요] 누락된 단계나 중복된 책임자를 점검하십시오.")

def analyze_mission4_dryrun(log_df):
    """미션 4 1차 모의훈련 타임라인 및 준수율 분석"""
    print("=" * 60)
    print("⏱️ [미션 4 분석] 1차 Dry-run 훈련 타임라인 및 SOP 준수율 계산")
    print("=" * 60)
    
    avg_lead_time = log_df['lead_time_sec'].mean()
    sop_pass_rate = (log_df['sop_pass'].sum() / len(log_df)) * 100
    
    print(log_df[['event_id', 'detect_sec', 'report_sec', 'relay_sec', 'action_sec', 'lead_time_sec', 'sop_pass']])
    print("-" * 60)
    print(f"✔ 평균 조치 완료 시간: {avg_lead_time:.1f}초 (기준: 180초 이내) -> {'[PASS]' if avg_lead_time <= 180 else '[FAIL]'}")
    print(f"✔ SOP 단계별 준수율: {sop_pass_rate:.1f}% (기준: 80% 이상) -> {'[PASS]' if sop_pass_rate >= 80 else '[FAIL]'}")
    
    if avg_lead_time <= 180 and sop_pass_rate >= 80:
        print("\n🎉 [미션 4 PASS] 1차 Dry-run 목표치를 달성하였습니다! 병목 구간 2건을 AAR에 인계하십시오.")

def grade_mission5_cross_verification(user_answers, answer_key):
    """미션 5 교차확인 판정 채점"""
    print("=" * 60)
    print("🎯 [미션 5 채점] 오류 주입 교차확인 판정 결과 검증")
    print("=" * 60)
    
    correct = 0
    miss_threat = 0
    total = len(answer_key)
    
    for card, info in answer_key.items():
        user_pred = user_answers.get(card, {}).get("pred")
        truth = info["truth"]
        is_threat = ("실제 위협" in truth)
        
        is_correct = (user_pred == truth)
        if is_correct:
            correct += 1
        elif is_threat and ("오경보" in str(user_pred)):
            miss_threat += 1
            print(f"🚨 [치명적 오류] {card} 위협을 오경보로 놓쳤습니다! (Miss 발생)")
            
    success_rate = (correct / total) * 100
    print(f"✔ 허위 경보 식별 성공률: {success_rate:.1f}% ({correct}/{total}) (기준: 85% 이상)")
    print(f"✔ 실제 위협 미탐(Miss) 건수: {miss_threat}건 (기준: 0건)")
    
    if success_rate >= 85.0 and miss_threat == 0:
        print("\n🎉 [미션 5 PASS] 교차 확인을 통해 적 기만 및 허위 경보를 완벽히 필터링했습니다!")

def compare_mission9_retrial(m4_data, m9_data):
    """미션 9 1차 대비 2차 훈련 개선율 분석"""
    print("=" * 60)
    print("📈 [미션 9 성과 분석] 1차(M4) 대비 2차 재훈련 반응 속도 & 오류 개선율")
    print("=" * 60)
    
    time1 = m4_data['avg_time']
    time2 = m9_data['avg_time']
    time_reduction = ((time1 - time2) / time1) * 100
    
    err1 = m4_data['errors']
    err2 = m9_data['errors']
    err_reduction = ((err1 - err2) / err1) * 100
    
    print(f"1. 전체 조치 소요 시간: {time1:.1f}초 ➔ {time2:.1f}초 (단축률: {time_reduction:.1f}% / 목표 30% 이상)")
    print(f"2. SOP 절차 오류 건수: {err1}건 ➔ {err2}건 (감소율: {err_reduction:.1f}% / 목표 50% 이상)")
    print("-" * 60)
    
    pass_time = time_reduction >= 30.0
    pass_err = err_reduction >= 50.0
    
    if pass_time and pass_err:
        print("🎉 [미션 9 PASS] 수정 SOP 적용을 통해 팀 단위 전술 반응 속도가 입증되었습니다!")


---

## 🎯 미션 01. 야전 부대 드론 경보 상황 분석 및 관측자-지휘자-경계조 역할 정의

### 🤖 AI Co-pilot 협업 가이드 (프롬프트 질의 및 검증)
> **💡 프롬프트 질의 예시:**
> `"야전 경계진지 드론 출현 가상 사례 C1~C6(정찰, 자폭, 통신두절 등)의 맥락을 분석하여, 최초 포착부터 전파 지연까지 발생한 핵심 전술적 문제점을 1줄 요약 표로 정리해줘."`
>
> **🔍 AI 결과 검증 포인트 (HITL):**
> - AI가 군사 보안 규정을 준수하며, 개인 실수가 아닌 7단계 지휘통제 절차 결함 관점으로 분석했는지 검증합니다.

---

### 📑 실습자료 및 과업 지침

야전 부대 드론 경보 상황 분석 및 관측자-지휘자-경계조 역할 정의**

가. 제공 자료 - 가상 드론 출현 사례 카드 [교육용가상자료]

| **카드** | **가상 시각·기상** | **상황** | **드러난 문제** |
| --- | --- | --- | --- |
| **C1** | 05:42, 옅은 안개 | OP-1 관측자가 윙윙거리는 소리를 들었으나 방향을 특정하지 못하고 '뭔가 소리가 난다'고만 보고했다. | 방위 없는 보고, 지휘자 되묻기 2회로 보고 지연 약 2분 |
| **C2** | 11:10, 맑음 | 12시 방향 나무 높이 위에서 드론 1기가 선회했다. 관측자는 즉시 보고했으나 경계조가 대피 위치를 몰라 우왕좌왕했다. | 대피 위치 사전 지정 없음 |
| **C3** | 14:25, 흐림 | 조류 무리를 드론으로 오인해 경보를 발령했고 경계조 전원이 대피해 5분간 경계 공백이 생겼다. | 교차 확인 없이 경보, 전원 대피로 경계 공백 |
| **C4** | 19:50, 해질녘 | 드론 2기가 접근했으나 관측자는 1기만 보고했다. 두 번째 기체는 경계조가 봤지만 누구에게 보고할지 몰랐다. | 경계조의 보고 경로 불명확 |
| **C5** | 23:30, 야간 | 무전 잡음 속에서 방위 '3시'가 '9시'로 전달되어 반대 방향을 경계했다. | 복창 확인 절차 없음 |
| **C6** | 02:15, 야간 | 지휘자가 순찰 중이라 부재했고 차선임자가 결심해도 되는지 몰라 대응이 지연되었다. | 지휘 위임 순서 미지정 |

나. 작성 서식 1 - 역할 분담 매트릭스 (수: 수행, 책: 책임, 협: 협조, 통: 통보 / 단계마다 '책'은 1명)

| **단계** | **관측자** | **지휘자** | **경계조** | **통신·기록** | **완료 판단 기준** |
| --- | --- | --- | --- | --- | --- |
| **① 포착** | 책·수 (예시) | 통 (예시) | 협 (예시) | - | 포착 시각 기록 (예시) |
| **② 식별** | ( ) | ( ) | ( ) | ( ) | ( ) |
| **③ 보고** | ( ) | ( ) | ( ) | ( ) | ( ) |
| **④ 전파** | ( ) | ( ) | ( ) | ( ) | ( ) |
| **⑤ 결심** | ( ) | ( ) | ( ) | ( ) | ( ) |
| **⑥ 대피·대응** | ( ) | ( ) | ( ) | ( ) | ( ) |
| **⑦ 해제** | ( ) | ( ) | ( ) | ( ) | ( ) |

다. 작성 서식 2 - 현장 체크 항목 정의표 (15개 이상)

| **번호** | **단계** | **체크 항목(눈으로 확인 가능한 문장)** | **확인 방법** | **담당** |
| --- | --- | --- | --- | --- |
| **1** | 포착 | 관측자가 담당 관측 구역을 말로 설명할 수 있다 (예시) | 평가관 질문 (예시) | 관측자 (예시) |
| **2** | 보고 | 1차 보고에 방위가 포함되었다 (예시) | 기록표 확인 (예시) | 지휘자 (예시) |
| **3** |  |  |  |  |
| **4** |  |  |  |  |
| **5** |  |  |  |  |
| **6** |  |  |  |  |
| **7** |  |  |  |  |
| **8** |  |  |  |  |
| **9** |  |  |  |  |
| **10** |  |  |  |  |
| **11** |  |  |  |  |
| **12** |  |  |  |  |
| **13** |  |  |  |  |
| **14** |  |  |  |  |
| **15** |  |  |  |  |
| **16** |  |  |  |  |
| **17** |  |  |  |  |

라. 작성 서식 3 - 부대 특성별 환경 변수 보강표

| **환경 변수** | **야전 진지** | **기지(주둔지)** | **의무시설** |
| --- | --- | --- | --- |
| **관측 시야(차폐물)** | ( ) | ( ) | ( ) |
| **대피·은폐 장소** | ( ) | ( ) | ( ) |
| **통신 수단** | ( ) | ( ) | ( ) |
| **경보 전파 대상** | ( ) | ( ) | ( ) |
| **소음 환경(청음 난이도)** | ( ) | ( ) | ( ) |
| **특별 고려 사항** | ( ) | ( ) | ( ) |

마. AI 제안 검토 기록

| **AI가 제안한 항목** | **채택 여부** | **채택·미채택 이유** |
| --- | --- | --- |
| **제안 1** |  |  |
| **제안 2** |  |  |
| **제안 3** |  |  |
| **제안 4** |  |  |

| **점검 항목** | **확인** |
| --- | --- |
| 사례 카드 6건의 문제점을 모두 역할 정의에 반영했다 | ( ) |
| 7단계 모두 역할 분담 매트릭스 빈칸이 없다 | ( ) |
| 단계마다 '책'이 1명만 지정되었다 | ( ) |
| 현장 체크 항목이 15개 이상이며 모두 눈으로 확인 가능한 문장이다 | ( ) |
| 환경 변수표 3개 부대 유형을 모두 채웠다 | ( ) |
| AI 제안의 채택 여부와 이유를 기록했다 | ( ) |
| 실제 부대명·위치·호출부호를 쓰지 않았다 | ( ) |

**



In [ ]:
# [미션 1 실습 코드] 학습자가 AI와 함께 작성한 RACI 매트릭스 및 체크리스트 입력
# TODO: AI 질의를 통해 완성한 7단계 RACI 딕셔너리와 체크리스트 15개를 채우고 실행하세요.
raci_data = {
    '포착': {'관측자': '수·책', '지휘자': '통', '경계조': '협'},
    # TODO: 식별, 보고, 전파, 결심, 대피·대응, 해제 단계를 채우세요 (단계별 책임자 '책'은 1명만)
}

checklist_items = [
    # TODO: 눈으로 확인 가능한 현장 체크 항목 15개 이상을 리스트로 입력하세요.
    '1. 관측 구역 방위 식별 가능 여부',
    '2. 1차 무전 보고 내 방위각 포함 여부'
]

# verify_mission1_raci(raci_data, checklist_items)


---

## 🎯 미션 02. 관측자(식별/보고), 지휘자(전파/결심), 경계조(대피/사격) 간 표준 행동 지침서 작성

### 🤖 AI Co-pilot 협업 가이드 (프롬프트 질의 및 검증)
> **💡 프롬프트 질의 예시:**
> `"M1 역할분담 매트릭스를 바탕으로 관측자·지휘자·경계조의 단계별 임무를 "신호-행동-보고" 3요소 표로 변환하고, 8대 필수 안전/보안 수칙을 적절한 단계에 배치해줘."`
>
> **🔍 AI 결과 검증 포인트 (HITL):**
> - 사격 전 지휘자 결심 및 교전수칙(ROE) 확인 단계가 명시되었는지, 모호한 표현(신속히/적절히)이 정량화되었는지 검토합니다.

---

### 📑 실습자료 및 과업 지침

관측자(식별/보고), 지휘자(전파/결심), 경계조(대피/사격) 간 표준 행동 지침서 작성**

가. 제공 자료 - 필수 안전/보안 조치 8개 [교육용가상자료]

| **번호** | **필수 조치** | **배치할 단계(작성)** |
| --- | --- | --- |
| **S1** | 경보 수신 즉시 지정된 은폐 위치로 이동한다(긴급 모드). | ( ) |
| **S2** | 사격은 지휘자 결심 후에만 한다. 훈련에서는 구두 선언으로 대체한다. | ( ) |
| **S3** | 사격 전 사격 방향에 아군·민간인이 없는지 확인한다(오사격 방지). | ( ) |
| **S4** | 무전은 짧게 하고 평문으로 진지 위치를 말하지 않는다. | ( ) |
| **S5** | 대피 후 지휘자는 인원을 확인(점호)하고 결과를 보고한다. | ( ) |
| **S6** | 부상자가 생기면 즉시 보고하고 응급처치·의무 인원에 연계한다. | ( ) |
| **S7** | 경보 해제 명령 전에는 임의로 원위치로 복귀하지 않는다. | ( ) |
| **S8** | 경보 중 개인 휴대전화·전자기기 사용과 촬영을 금지한다. | ( ) |

나. 제공 자료 - 모호 표현 바꾸기 예시

| **모호한 표현** | **구체적 기준(예시)** |
| --- | --- |
| **신속히 보고한다** | 포착 후 30초 이내에 방위·수량만이라도 1차 보고한다 |
| **적절한 위치로 대피한다** | 사전 지정한 은폐 위치 ○번(가상)으로 이동한다 |
| **상황을 판단해 조치한다** | 2명 이상 확인되면 경보 발령, 1명만 확인되면 '확인 요망' 전파 |
| **필요시 사격한다** | 지휘자의 '사격' 명령이 있을 때만 구두 사격 선언 |

다. 작성 서식 1 - 역할별 표준 행동 지침(SOP) 표 (신호 - 행동 - 보고)

| **단계** | **신호(무엇을 보거나 들으면)** | **관측자** | **지휘자** | **경계조** | **안전 조치(S번호)** |
| --- | --- | --- | --- | --- | --- |
| **① 포착** |  |  |  |  |  |
| **② 식별** |  |  |  |  |  |
| **③ 보고** |  |  |  |  |  |
| **④ 전파** |  |  |  |  |  |
| **⑤ 결심** |  |  |  |  |  |
| **⑥ 대피·대응** |  |  |  |  |  |
| **⑦ 해제** |  |  |  |  |  |

라. 작성 서식 2 - SOP 흐름도 틀 (빈칸을 채워 완성하고, 긴급/일반 분기를 표시한다)

| [① 포착: 소리 또는 육안]
 \|
 < 즉시 위협인가? (근접·급강하·다수) >
 \|예 \|아니오
 [긴급 모드] [일반 모드]
 전원 ( ) 관측자 ( )
 지휘자 ( ) 지휘자 복창·교차 확인 ( )
 \| \|
 +------------+---------------+
 \|
 [⑤ 결심: ( )]
 \|
 [⑥ 대피·대응: ( )]
 \|
 [⑦ 해제: 해제 명령 → 인원 확인 → ( )] |
| --- |

마. 작성 서식 3 - 휴대용 요약 카드(Pocket Guide, 명함 2장 크기로 줄여 인쇄)

| **구분** | **관측자** | **지휘자** | **경계조** |
| --- | --- | --- | --- |
| **긴급 모드 첫 행동** | ( ) | ( ) | ( ) |
| **일반 모드 첫 행동** | ( ) | ( ) | ( ) |
| **반드시 말할 것** | ( ) | ( ) | ( ) |
| **절대 하지 말 것** | ( ) | ( ) | ( ) |

바. 교차 검토 기록표 (다른 조가 작성)

| **번호** | **지적 위치** | **지적 내용(누락·모호)** | **수정 결과** | **확인** |
| --- | --- | --- | --- | --- |
| **1** |  |  |  |  |
| **2** |  |  |  |  |
| **3** |  |  |  |  |
| **4** |  |  |  |  |
| **5** |  |  |  |  |

| **점검 항목** | **확인** |
| --- | --- |
| 7단계 모두 관측자·지휘자·경계조 행동이 적혔다(빈칸 0) | ( ) |
| 필수 안전/보안 조치 S1~S8이 모두 배치되었다 | ( ) |
| 사격 행동은 지휘자 결심 이후에만 나온다 | ( ) |
| 긴급/일반 모드가 흐름도에서 구분된다 | ( ) |
| 교차 검토 지적 사항을 모두 수정했다 | ( ) |
| 요약 카드가 한 장 안에 들어간다 | ( ) |

**



---

## 🎯 미션 03. 무전 약어 및 육안 식별 보고 템플릿(방위, 고도, 기종, 속도) 표준화

### 🤖 AI Co-pilot 협업 가이드 (프롬프트 질의 및 검증)
> **💡 프롬프트 질의 예시:**
> `"SALUTE 보고 체계를 드론 상황에 맞게 개편한 7개 필수 항목과 시계방향 방위, 상대고도 표현을 적용한 가상 무전 약어표 및 10대 MSEL 시나리오를 작성해줘."`
>
> **🔍 AI 결과 검증 포인트 (HITL):**
> - 정상 드론 6건과 오인/교란 4건(6:4 비율)이 안배되었는지, 무전 발성 시 혼선이 없는지 점검합니다.

---

### 📑 실습자료 및 과업 지침

무전 약어 및 육안 식별 보고 템플릿(방위, 고도, 기종, 속도) 표준화**

가. 제공 자료 - 드론 보고 7항목 표현 기준(초안) [교육용가상자료]

| **항목** | **표현 방법** | **예시** | **보완 의견(작성)** |
| --- | --- | --- | --- |
| **방위** | 진지 정면을 12시로 한 시계 방향 | 2시 방향 | ( ) |
| **고도** | 지형지물 기준 상대 표현(지면 가까이 / 나무 높이 / 나무 위 / 높음) | 나무 위 | ( ) |
| **기종** | 회전익 / 고정익 / 미상 | 회전익 | ( ) |
| **속도** | 정지비행 / 느림 / 빠름 | 느림 | ( ) |
| **수량** | 기체 수, 불확실하면 '최소 ○기' | 최소 2기 | ( ) |
| **행동** | 접근 / 선회 / 이탈 / 정지 / 급강하 | 접근 중 | ( ) |
| **시각** | 최초 포착 시각(시·분) | 05시 42분 | ( ) |

나. 제공 자료 - 교육용 가상 무전 약어표 (실제 음어·호출부호 사용 금지)

| **약어(가상)** | **뜻** | **사용 예** | **수정·추가(작성)** |
| --- | --- | --- | --- |
| **드경** | 드론 경보(위협 확인) | 관측1, 드경, 2시 나무 위 회전익 1기 | ( ) |
| **확요** | 확인 요망(위협 미확인) | 관측2, 확요, 9시 소리만 | ( ) |
| **복창** | 받은 내용을 그대로 되풀이 | 지휘, 복창, 2시 회전익 1기 | ( ) |
| **은폐** | 즉시 은폐 위치로 이동 | 지휘, 전원 은폐 | ( ) |
| **해제** | 경보 해제 | 지휘, 드경 해제, 인원 보고 | ( ) |
|  |  |  |  |
|  |  |  |  |

다. 제공 자료 - 7항목 보고 문안 예시

| 관측1, 드경. 2시 방향, 나무 위, 회전익 1기, 느림, 접근 중, 05시 42분. 이상.
지휘, 복창. 2시 나무 위 회전익 1기 접근. 전원 은폐. 관측1 계속 추적 보고. |
| --- |

라. 작성 서식 - 상황 주입 시나리오(MSEL) (10건 이상, 정상:비정상 = 6:4)

| **번호** | **가상 시각** | **주입 수단** | **사건 내용(주입 메시지 문안)** | **구분** | **기대 행동** | **담당** |
| --- | --- | --- | --- | --- | --- | --- |
| **1** | 05:40 | 구두 | OP-1 방향에서 멀리 윙윙 소리 (예시) | 정상 | 확요 보고 → 지휘자 복창 (예시) | 관측자 |
| **2** | 05:43 | 카드 | 2시 나무 위 회전익 1기 접근 (예시) | 정상 | 드경 7항목 보고 → 은폐 (예시) | 전원 |
| **3** | 05:47 | 카드 | 9시 방향 새 떼 이동 (예시) | 비정상 | 교차 확인 후 오경보 판정 (예시) | 지휘자 |
| **4** |  |  |  |  |  |  |
| **5** |  |  |  |  |  |  |
| **6** |  |  |  |  |  |  |
| **7** |  |  |  |  |  |  |
| **8** |  |  |  |  |  |  |
| **9** |  |  |  |  |  |  |
| **10** |  |  |  |  |  |  |
| **11** |  |  |  |  |  |  |
| **12** |  |  |  |  |  |  |

| **구성 비율 계산** | 정상 ( )건 : 비정상 ( )건 = ( ) : ( ) → 6:4 충족 여부 ( ) |
| --- | --- |

마. 워크스루(소리 내어 따라가기) 점검표

| **점검 질문** | **문제 발견 사건 번호** | **수정 내용** |
| --- | --- | --- |
| **사건 순서와 간격이 현실적인가(너무 몰리거나 비지 않는가)?** |  |  |
| **모든 사건에서 7항목 보고가 가능한가(빠지는 항목은?)** |  |  |
| **비정상 사건이 너무 쉽게 드러나지 않는가?** |  |  |
| **돌발 변수(통신 두절·센서 오류·부상)가 1건 이상 있는가?** |  |  |

| **점검 항목** | **확인** |
| --- | --- |
| 보고 템플릿에 7항목이 모두 있다 | ( ) |
| 약어는 가상 표현만 사용했다 | ( ) |
| 주입 사건이 10건 이상이다 | ( ) |
| 정상/비정상 비율이 6:4다 | ( ) |
| 돌발 변수가 1건 이상 포함되었다 | ( ) |
| 워크스루 지적 사항을 모두 수정했다 | ( ) |

**



---

## 🎯 미션 04. 가상 드론 경보 발령 시 역할별 초동 조치 및 보고 체계 1차 모의훈련(Dry-run)

### 🤖 AI Co-pilot 협업 가이드 (프롬프트 질의 및 검증)
> **💡 프롬프트 질의 예시:**
> `"1차 Dry-run 10개 사건의 초 단위 타임스탬프 기록을 바탕으로 평균 지연 시간과 SOP 준수율(%)을 계산하고, 가장 심각한 병목 구간 2건을 도출해줘."`
>
> **🔍 AI 결과 검증 포인트 (HITL):**
> - 타임스탬프 계산이 원시 데이터와 일치하는지 교차 검산하고 구조적 통신 병목 원인을 검증합니다.

---

### 📑 실습자료 및 과업 지침

가상 드론 경보 발령 시 역할별 초동 조치 및 보고 체계 1차 모의훈련(Dry-run)**

가. 제공 자료 - 1차 모의훈련 진행 시간표(권장)

| **시간** | **활동** | **담당** |
| --- | --- | --- |
| **0~10분** | 역할 카드 배부, 안전 브리핑, 초시계 동기화 | 교관 |
| **10~15분** | SOP·요약 카드 개인 숙지 | 전원 |
| **15~45분** | 주입 카드 1~10번 순서 진행(1차 Dry-run) | 교관·전원 |
| **45~60분** | 기록표 정리, 준수율·지연시간 계산 | 기록자·평가관 |
| **60~75분** | 병목·혼선 구간 토의 | 전원 |

나. 작성 서식 1 - 시각 기록표 (기록자, 초 단위)

| **카드** | **최초 포착** | **1차 보고** | **지휘자 수신** | **전파** | **대피 완료** | **보고 지연(초)
=수신-포착** | **조치 시간(초)
=대피-포착** |
| --- | --- | --- | --- | --- | --- | --- | --- |
| **예시** | 05:43:05 | 05:43:32 | 05:43:40 | 05:43:55 | 05:44:30 | 35 | 85 |
| **1** |  |  |  |  |  |  |  |
| **2** |  |  |  |  |  |  |  |
| **3** |  |  |  |  |  |  |  |
| **4** |  |  |  |  |  |  |  |
| **5** |  |  |  |  |  |  |  |
| **6** |  |  |  |  |  |  |  |
| **7** |  |  |  |  |  |  |  |
| **8** |  |  |  |  |  |  |  |
| **9** |  |  |  |  |  |  |  |
| **10** |  |  |  |  |  |  |  |

다. 작성 서식 2 - SOP 단계 수행 체크(평가관)

| **카드** | **① 포착** | **② 식별** | **③ 보고** | **④ 전파** | **⑤ 결심** | **⑥ 대피·대응** | **⑦ 해제** | **수행/요구** |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| **1** |  |  |  |  |  |  |  |  |
| **2** |  |  |  |  |  |  |  |  |
| **3** |  |  |  |  |  |  |  |  |
| **4** |  |  |  |  |  |  |  |  |
| **5** |  |  |  |  |  |  |  |  |
| **6** |  |  |  |  |  |  |  |  |
| **7** |  |  |  |  |  |  |  |  |
| **8** |  |  |  |  |  |  |  |  |
| **9** |  |  |  |  |  |  |  |  |
| **10** |  |  |  |  |  |  |  |  |

| **SOP 준수율** | 수행한 단계 합계 ( ) ÷ 요구된 단계 합계 ( ) × 100 = ( )% → 목표 80% 이상 ( ) |
| --- | --- |

| **평균 보고 지연** | 보고 지연 합계 ( )초 ÷ 카드 수 ( ) = ( )초 → 목표 180초(3분) 이내 ( ) |
| --- | --- |

라. 작성 서식 3 - 참여자별 수행 기록지(각자 작성)

| **역할** | **잘한 점** | **막힌 점(카드 번호)** | **SOP에 없어서 헷갈린 것** |
| --- | --- | --- | --- |
| **관측자** |  |  |  |
| **지휘자** |  |  |  |
| **경계조** |  |  |  |
| **통신·기록** |  |  |  |

마. 작성 서식 4 - 병목·혼선 구간 기록

| **번호** | **발생 단계·카드** | **현상** | **추정 원인** | **M8로 넘길 질문** |
| --- | --- | --- | --- | --- |
| **1** |  |  |  |  |
| **2** |  |  |  |  |
| **3** |  |  |  |  |

| **점검 항목** | **확인** |
| --- | --- |
| 모든 카드의 시각이 초 단위로 기록되었다 | ( ) |
| SOP 준수율을 계산식대로 계산했다 | ( ) |
| 평균 보고 지연을 계산했다 | ( ) |
| 참여자 전원의 수행 기록지를 수합했다 | ( ) |
| 병목·혼선 구간을 2건 이상 적었다 | ( ) |
| 훈련 중 안전 위반이 없었다 | ( ) |

**



In [ ]:
# [미션 4 실습 코드] 모의훈련 10개 사건의 초 단위 기록 데이터 입력 및 분석
# TODO: 모의훈련 관측 기록표의 데이터를 DataFrame으로 입력하고 분석 함수를 실행하세요.
data = [
    {'event_id': 'MSEL-01', 'detect_sec': 5, 'report_sec': 24, 'relay_sec': 48, 'action_sec': 78, 'lead_time_sec': 78, 'sop_pass': True},
    # TODO: MSEL 02부터 10까지 10건의 측정 결과를 채우세요
]
# log_df = pd.DataFrame(data)
# analyze_mission4_dryrun(log_df)


---

## 🎯 미션 05. 잘못된 방위 보고 및 통신 지연 등 인위적 오류 주입 시 교차 확인 훈련

### 🤖 AI Co-pilot 협업 가이드 (프롬프트 질의 및 검증)
> **💡 프롬프트 질의 예시:**
> `"무작위 12개 사건(E01~E12)에 대해 6대 교차확인 규칙을 적용하여 실제 위협과 오경보를 분류하고 그 전술적 근거를 교차검증 매트릭스로 정리해줘."`
>
> **🔍 AI 결과 검증 포인트 (HITL):**
> - 실제 위협을 오경보로 놓친 미탐(Miss)이 0건인지 엄격히 검증하고, 판정 근거의 타당성을 확인합니다.

---

### 📑 실습자료 및 과업 지침

잘못된 방위 보고 및 통신 지연 등 인위적 오류 주입 시 교차 확인 훈련**

가. 제공 자료 - 교차 확인 규칙(초안, 조별로 보완)

| **규칙** | **내용** |
| --- | --- |
| **R1 복창** | 지휘자는 받은 보고의 방위·수량을 복창하고 관측자가 '맞음/틀림'으로 답한다. |
| **R2 2차 관측** | 가능하면 다른 관측자에게 같은 방위를 확인시킨다. |
| **R3 시각 대조** | 보고 시각과 수신 시각이 1분 이상 차이 나면 지연 정보로 표시한다. |
| **R4 소리-육안 일치** | 소리 방향과 육안 방향이 다르면 '확인 요망'으로 둔다. |
| **R5 송신자 확인** | 송신자를 알 수 없는 보고는 되물어 확인한다. |
| **R6 안전 우선** | 확인 중이라도 급강하·근접 신호가 있으면 먼저 은폐를 전파한다. |

나. 제공 자료 - 오류 주입 카드 12장 [교육용가상자료] (교관이 순서를 섞어 제시)

| **카드** | **들어온 보고** | **추가로 확인되는 정보** |
| --- | --- | --- |
| **E1** | 관측1: 2시 나무 위 회전익 1기 접근 | 관측2도 2시에서 같은 기체 확인 |
| **E2** | 관측2: 9시 드론 1기 (실제는 3시) | 관측1은 3시에서 소리 청취, 무전 잡음 있음 |
| **E3** | 관측1: 12시 높은 곳 일정 속도 비행체, 소리 약함 | 일정한 직선 궤적, 깜박이는 불빛 |
| **E4** | 경계조: 10시 드론 무리 여러 기 | 날갯짓, 방향이 수시로 바뀜, 소리 없음 |
| **E5** | 관측2: 4시 회전익 1기 정지비행 | 관측1도 4시에서 윙윙 소리 확인 |
| **E6** | 관측1: 2시 회전익 1기 (3분 전 E1과 같은 내용이 늦게 도착) | 무전 지연, 시각 표기가 3분 전 |
| **E7** | 관측2: 7시 드론 2기 접근 | 관측1은 7시에서 1기만 확인, 이후 2기째 확인 |
| **E8** | 관측1: 11시 흰색 물체가 천천히 떠감 | 바람 방향과 같은 방향으로 표류, 소리 없음 |
| **E9** | 경계조: 5시 윙윙 소리, 육안 미확인 | 관측2가 5시에서 저고도 회전익 1기 육안 확인 |
| **E10** | 관측1·관측2 동시: 1시 드론 1기, 1시 드론 1기 | 같은 시각·같은 방위(중복 보고) |
| **E11** | 관측2: 8시 급강하 기체 | 관측1 확인 불가(시야 차단), 소리 급격히 커짐 |
| **E12** | 무전: '6시 드론 3기' (송신자 불명) | 관측자·경계조 누구도 송신하지 않음, 6시 방향 조용함 |

다. 작성 서식 1 - 교차검증 매트릭스

| **카드** | **적용 규칙(R번호)** | **판정
(위협/오경보/확인필요)** | **근거(관측 사실)** | **지휘자 조치** |
| --- | --- | --- | --- | --- |
| **E1** |  |  |  |  |
| **E2** |  |  |  |  |
| **E3** |  |  |  |  |
| **E4** |  |  |  |  |
| **E5** |  |  |  |  |
| **E6** |  |  |  |  |
| **E7** |  |  |  |  |
| **E8** |  |  |  |  |
| **E9** |  |  |  |  |
| **E10** |  |  |  |  |
| **E11** |  |  |  |  |
| **E12** |  |  |  |  |

| **식별 성공률** | 정답과 일치한 카드 ( ) ÷ 전체 12장 × 100 = ( )% → 목표 85% 이상 ( ) |
| --- | --- |

| **위협 놓침** | 실제 위협을 오경보로 판정한 건수 ( )건 → 목표 0건 ( ) |
| --- | --- |

라. 작성 서식 2 - 불일치 시 재확인·보고 경로(우선순위 규칙)

| **상황** | **1순위 확인** | **2순위 확인** | **상급 보고 여부·시점** |
| --- | --- | --- | --- |
| **두 관측자의 방위가 다름** |  |  |  |
| **보고가 늦게 도착(1분 이상)** |  |  |  |
| **송신자 불명 보고** |  |  |  |
| **소리만 있고 육안 미확인** |  |  |  |

마. 교관용 정답표 (훈련 후 공개)

| **카드** | **정답** | **핵심 근거** |
| --- | --- | --- |
| **E1** | 위협 | 두 관측자 방위 일치 |
| **E2** | 위협(방위 정정 3시) | 잡음, 다른 관측자 3시 청취 → 복창으로 정정 |
| **E3** | 오경보 | 높은 고도·일정 궤적·깜박이는 불빛 → 민간 항공기 추정 |
| **E4** | 오경보 | 날갯짓·방향 급변·무소음 → 조류 |
| **E5** | 위협 | 소리·육안 일치 |
| **E6** | 오경보(지연 중복) | 3분 전 E1 정보의 지연 도착 |
| **E7** | 위협(2기) | 추가 확인으로 2기 확정 |
| **E8** | 오경보 | 바람 따라 표류·무소음 → 비닐·풍선류 |
| **E9** | 위협 | 관측2 육안 확인 |
| **E10** | 위협(1기, 중복 보고) | 같은 시각·방위 → 1기로 정리 |
| **E11** | 위협(긴급) | 급강하·소리 급증 → R6 안전 우선 은폐 |
| **E12** | 확인필요→오경보 | 송신자 불명, 현장 무신호 → 되물어 확인 후 기만 가능성 보고 |

| 판정 점수 기준: '확인필요'로 두고 이후 확인을 거쳐 정답에 도달했으면 정답으로 본다. 실제 위협(E1·E2·E5·E7·E9·E10·E11)을 오경보로 버리면 성공률과 관계없이 FAIL이다. |
| --- |

| **점검 항목** | **확인** |
| --- | --- |
| 12장 모두 판정과 근거를 적었다 | ( ) |
| 근거는 관측 사실(방위·소리·궤적·시각)로 적었다 | ( ) |
| 식별 성공률과 위협 놓침 건수를 계산했다 | ( ) |
| 불일치 시 보고 경로 4가지 상황을 채웠다 | ( ) |
| 급강하·근접 신호에서는 확인보다 은폐 전파를 먼저 했다 | ( ) |

**



In [ ]:
# [미션 5 실습 코드] 사건 카드 E01~E12 조별 판정 결과 입력
# TODO: 교차확인 규칙 R1~R6을 적용하여 내린 판정 결과를 딕셔너리로 입력하세요.
user_answers = {
    'E01': {'pred': '실제 위협'},
    'E02': {'pred': '오경보 (조류)'},
    # TODO: E03 ~ E12 판정 결과를 채우세요
}
# grade_mission5_cross_verification(user_answers, answer_key)


---

## 🎯 미션 06. 대공 사격 구역 할당 및 인접 분대 상황 전파 인계 실습

### 🤖 AI Co-pilot 협업 가이드 (프롬프트 질의 및 검증)
> **💡 프롬프트 질의 예시:**
> `"가상 진지 4대 구역의 사격 구역 중복과 사각지대를 조정한 재할당 표와 SBAR 10개 필수 항목 인계 브리핑 문안을 작성해줘."`
>
> **🔍 AI 결과 검증 포인트 (HITL):**
> - 구역 중복·공백이 0건인지, SBAR 브리핑이 10분 내에 전달 가능한 명확한 핵심 정보로 구성되었는지 검증합니다.

---

### 📑 실습자료 및 과업 지침

대공 사격 구역 할당 및 인접 분대 상황 전파 인계 실습**

가. 제공 자료 - 가상 진지 구역표(추상화, 실제 배치 아님) [교육용가상자료]

| **구역** | **방향(진지 정면 12시 기준)** | **지형 특징(가상)** | **관측 난이도** |
| --- | --- | --- | --- |
| **A** | 12시~3시 | 개활지, 시야 넓음 | 낮음 |
| **B** | 3시~6시 | 숲 가장자리, 저고도 접근 시 늦게 보임 | 높음 |
| **C** | 6시~9시 | 후방 도로, 아군 이동 잦음 | 보통(아군 오인 주의) |
| **D** | 9시~12시 | 낮은 능선, 역광 시간대 있음 | 보통 |

나. 제공 자료 - 분대별 감시·사격 구역 초안(오류 포함)

| **분대** | **초안 감시·사격 구역** | **조정안(작성)** | **조정 근거(작성)** |
| --- | --- | --- | --- |
| **1분대** | 11시~3시 |  |  |
| **2분대** | 2시~6시 |  |  |
| **3분대** | 7시~10시 |  |  |

| **중복 구간** | ( ) ※ 초안에서 찾은 겹치는 방향을 모두 적는다 |
| --- | --- |

| **공백 구간** | ( ) ※ 어느 분대도 맡지 않은 방향을 모두 적는다 |
| --- | --- |

| **경계 방향 처리 원칙** | ( ) ※ 예: 경계선 방향은 양쪽 분대가 공동 관측하고 사격은 한쪽만 담당 |
| --- | --- |

다. 제공 자료 - 인계 상황 카드(진행 중 상황) [교육용가상자료]

| 가상 05:55 현재: 드론 경보 발령 중. 05:43에 2시 방향 회전익 1기 접근 후 05:49 1시 방향으로 이탈, 05:52 4시 방향 소리만 청취(확인 요망 상태). 경계조 1명 발목 경상(모의, 응급처치 완료). 무전기 잡음 심함, 06:00 교대 예정. 상급 보고는 05:45 1회 완료, 다음 보고 예정 06:00. |
| --- |

라. 작성 서식 1 - SBAR 인계 브리핑 템플릿

| **구분** | **들어갈 내용** | **작성** |
| --- | --- | --- |
| **S 상황** | 현재 경보 단계, 지금 보고 있는 표적 |  |
| **B 배경** | 지금까지 경과(시각순), 이미 한 조치 |  |
| **A 평가** | 위협 판단, 불확실한 점, 인원·통신 상태 |  |
| **R 요청** | 인수자가 곧바로 할 일, 다음 보고 시각 |  |

마. 작성 서식 2 - 인계 필수 항목 10개 확인표

| **번호** | **필수 항목** | **전달(O/X)** | **인수자 복창(O/X)** |
| --- | --- | --- | --- |
| **1** | 현재 경보 단계 | ( ) | ( ) |
| **2** | 표적 수·방위·마지막 확인 시각 | ( ) | ( ) |
| **3** | 확인 요망 중인 신호 | ( ) | ( ) |
| **4** | 담당 감시·사격 구역 | ( ) | ( ) |
| **5** | 인원 현황·부상자 | ( ) | ( ) |
| **6** | 통신 상태 | ( ) | ( ) |
| **7** | 상급 보고 이력 | ( ) | ( ) |
| **8** | 다음 보고 예정 시각 | ( ) | ( ) |
| **9** | 인수자 즉시 조치 사항 | ( ) | ( ) |
| **10** | 인계·인수 시각 | ( ) | ( ) |

| **인계 소요 시간** | 시작 ( : : ) ~ 종료 ( : : ) = ( )분 → 목표 10분 이내 ( ) |
| --- | --- |

| **누락률** | 누락 항목 수 ( ) ÷ 10 × 100 = ( )% → 목표 0% ( ) |
| --- | --- |

| **상호 서명** | 인계자 ( ) 인수자 ( ) |
| --- | --- |

바. 인수자 돌발 상황 퀴즈 (교관이 인계 직후 구두로 질문)

| **번호** | **질문** | **정답(교관용)** | **O/X** |
| --- | --- | --- | --- |
| **Q1** | 지금 확인 요망 상태인 신호는 어느 방향인가? | 4시 방향(소리만) | ( ) |
| **Q2** | 마지막으로 육안 확인된 드론은 어디로 이탈했나? | 1시 방향 | ( ) |
| **Q3** | 부상자는 몇 명이고 상태는? | 1명, 발목 경상, 응급처치 완료 | ( ) |
| **Q4** | 다음 상급 보고 시각은? | 06:00 | ( ) |
| **Q5** | 지금 4시 방향에서 윙윙 소리가 커지면 가장 먼저 할 일은? | 은폐 전파 후 7항목 보고 | ( ) |

| **점검 항목** | **확인** |
| --- | --- |
| 구역 중복·공백이 0개가 되도록 조정했다 | ( ) |
| 조정 근거에 지형·관측 난이도를 반영했다 | ( ) |
| SBAR 4개 칸을 모두 채웠다 | ( ) |
| 필수 항목 10개 누락 0 | ( ) |
| 인계를 10분 이내에 마쳤다 | ( ) |
| 인수자 퀴즈 4문항 이상 정답 | ( ) |
| 상호 서명을 받았다 | ( ) |

**



---

## 🎯 미션 07. 야간 기습 및 다수 드론 동시 출현 극한 상황 모의 역할분담 스트레스 훈련

### 🤖 AI Co-pilot 협업 가이드 (프롬프트 질의 및 검증)
> **💡 프롬프트 질의 예시:**
> `"야간 조명 제한 및 다수 드론 동시 침투 스트레스 상황에서 지휘권 승계 순서와 10대 위험 방지 조치 이행 타임라인을 구성해줘."`
>
> **🔍 AI 결과 검증 포인트 (HITL):**
> - 비상 절차 전환이 60초 내 완료되었는지, 차선임자 지휘권 승계 선언이 무전망에 명확히 전파되었는지 검토합니다.

---

### 📑 실습자료 및 과업 지침

야간 기습 및 다수 드론 동시 출현 극한 상황 모의 역할분담 스트레스 훈련**

가. 제공 자료 - 스트레스 주입 타임라인 [교육용가상자료] (교관이 T 기준으로 제시)

| **시점** | **주입 내용** | **기대 대응** |
| --- | --- | --- |
| **T+0:00** | 야간, 조명 제한(손전등 사용 금지 가정) | 야간 관측 위치 확인, 요약 카드 확인 |
| **T+1:00** | 3시 방향 윙윙 소리 | 확요 보고, 지휘자 복창 |
| **T+2:00** | 1시·4시·8시에서 드론 3기 동시 출현 | 긴급 모드 전환, 전원 은폐, 표적 우선순위 보고 |
| **T+4:00** | 지휘자 무전 두절(지휘자 역할 퇴장) | 위임 순서에 따라 차선임자 지휘 인수 |
| **T+5:00** | 경계조 1명 넘어짐(모의 경상) | 부상 보고, 응급처치 연계, 인원 확인 |
| **T+7:00** | 인접 분대가 '8시 기체 확인 요청' | 구역 확인 후 관측 결과 회신 |
| **T+9:00** | 2시에서 소리 + 새 떼 소리 혼재 | 교차 확인, 오경보 여부 판정 |
| **T+12:00** | 지휘자 무전 복구 | 차선임자가 지휘자에게 상황 인계(SBAR) |
| **T+14:00** | 해제 명령 | 인원 확인 후 해제 보고 |

나. 제공 자료 - 위험 조치 10개 판정 기준

| **번호** | **위험 조치** | **성공 기준** |
| --- | --- | --- |
| **D1** | 긴급 모드 전환 | 3기 출현 후 1분 이내 '전원 은폐' 전파 |
| **D2** | 은폐 이동 | 전원이 지정 위치로 이동(평가관 확인) |
| **D3** | 표적 우선순위 보고 | 가장 가까운·급강하 표적부터 보고 |
| **D4** | 오사격 방지 확인 | 구두 사격 전 아군·민간인 확인 발언 |
| **D5** | 지휘 위임 | 두절 후 1분 이내 차선임자 '지휘 인수' 선언 |
| **D6** | 부상 보고 | 부상 발생 1분 이내 보고 |
| **D7** | 인원 확인 | 부상 후 및 해제 전 점호 실시 |
| **D8** | 인접 분대 회신 | 요청 2분 이내 관측 결과 회신 |
| **D9** | 오경보 판정 | 교차 확인 후 판정 근거 발언 |
| **D10** | 지휘 복귀 인계 | SBAR 4요소를 포함해 인계 |

다. 작성 서식 1 - 타임라인 기록표(기록자)

| **시점** | **실제 행동·발언** | **행동 시각** | **걸린 시간(초)** | **관련 D번호** |
| --- | --- | --- | --- | --- |
| **T1** |  |  |  |  |
| **T2** |  |  |  |  |
| **T3** |  |  |  |  |
| **T4** |  |  |  |  |
| **T5** |  |  |  |  |
| **T6** |  |  |  |  |
| **T7** |  |  |  |  |
| **T8** |  |  |  |  |
| **T9** |  |  |  |  |

라. 작성 서식 2 - 위험 조치 판정표 및 위임 순서

| **D1** | **D2** | **D3** | **D4** | **D5** | **D6** | **D7** | **D8** | **D9** | **D10** |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| ( ) | ( ) | ( ) | ( ) | ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |

| **위험 조치 성공률** | 성공 ( ) ÷ 10 × 100 = ( )% → 목표 90% 이상 ( ) |
| --- | --- |

| **비상 절차 전환 시간** | 3기 출현 ( : : ) → 은폐 전파 ( : : ) = ( )초 → 목표 60초 이내 ( ) |
| --- | --- |

| **지휘 위임 순서** | 1순위 지휘자 ( ) → 2순위 ( ) → 3순위 ( ) / 인수 선언 문구: ( ) |
| --- | --- |

마. 안전 통제 점검(훈련 시작 전, 안전 통제관)

| **안전 점검 항목** | **확인** |
| --- | --- |
| 이동 경로에 걸려 넘어질 물체를 치웠다 | ( ) |
| 야간 조명 제한 구역 밖 대기 인원을 지정했다 | ( ) |
| '훈련 중지' 3회 신호를 전원이 알고 있다 | ( ) |
| 모의 부상은 실제 동작 없이 카드로만 표시한다 | ( ) |
| 실탄·공포탄·실제 화기가 없음을 확인했다 | ( ) |

| **점검 항목** | **확인** |
| --- | --- |
| 타임라인 9개 시점을 모두 기록했다 | ( ) |
| 비상 절차 전환 시간을 초 단위로 계산했다 | ( ) |
| 위험 조치 10개를 모두 판정했다 | ( ) |
| 위임이 1분 이내에 이루어졌는지 확인했다 | ( ) |
| 안전 수칙 위반 0건 | ( ) |

**



---

## 🎯 미션 08. AAR 회의를 통한 보고 지연 및 사격 구역 중복 문제점 분석

### 🤖 AI Co-pilot 협업 가이드 (프롬프트 질의 및 검증)
> **💡 프롬프트 질의 예시:**
> `"참여자 진술과 실제 관측 로그(F1~F7)를 팩트체크하고, 4M(절차/도구/숙련/환경) 원인 분석을 통해 비난 없는 SOP 1차 개선점 5건을 도출해줘."`
>
> **🔍 AI 결과 검증 포인트 (HITL):**
> - 주관적 비난 표현이 배제되었는지, 현행-수정-근거 3단 표가 실제 훈련 결함 데이터에 부합하는지 검증합니다.

---

### 📑 실습자료 및 과업 지침

AAR 회의를 통한 보고 지연 및 사격 구역 중복 문제점 분석**

가. 제공 자료 - AAR 4단계 진행 순서(권장 60분)

| **단계** | **질문** | **시간** |
| --- | --- | --- |
| **1** | 무엇을 하려고 했나(계획·SOP 기준)? | 10분 |
| **2** | 실제로 무슨 일이 있었나(기록 기준)? | 15분 |
| **3** | 왜 차이가 났나(원인)? | 20분 |
| **4** | 다음에는 무엇을 바꿀까(개선)? | 15분 |

나. 제공 자료 - 가상 기록 발췌(팩트체크 연습용) [교육용가상자료]

| **번호** | **출처** | **내용** |
| --- | --- | --- |
| **F1** | 관측자 진술 | "드론을 보자마자 30초 안에 보고했다." |
| **F2** | 시각 기록표 | 최초 포착 05:43:05, 지휘자 수신 05:44:55 (110초) |
| **F3** | 통신 기록 | 05:43:20~05:44:40 사이 지휘자 무전이 다른 보고로 점유됨 |
| **F4** | 구역 할당표 | 1분대 11시~3시, 2분대 2시~6시 (2~3시 중복) |
| **F5** | 평가관 기록 | 2시 표적에 1분대·2분대가 동시에 '사격 준비' 구두 선언 |
| **F6** | 지휘자 진술 | "2분대가 2시를 맡는 줄 알았다." |
| **F7** | 스트레스 훈련 기록 | 지휘 위임 선언까지 95초 소요, 위임 문구가 정해져 있지 않았음 |

| 팩트체크 예: F1 진술(30초)과 F2 기록(110초)이 다르다. F3를 보면 관측자는 제때 송신하려 했으나 무전이 점유되어 있었다. 원인은 '관측자 태만'이 아니라 '보고 채널 우선순위 규칙 없음(절차)'으로 분류한다. |
| --- |

다. 작성 서식 1 - 원인 분석 매트릭스

| **문제** | **확인된 사실** | **근거(F번호·기록)** | **원인 분류
(절차/도구/환경/숙련)** | **근본 원인** | **개선안** |
| --- | --- | --- | --- | --- | --- |
| **보고 지연** |  |  |  |  |  |
| **사격 구역 중복** |  |  |  |  |  |
| **지휘 위임 지연** |  |  |  |  |  |
|  |  |  |  |  |  |
|  |  |  |  |  |  |

라. 비난 표현 바꾸기

| **피할 표현** | **바꾼 표현(예시)** |
| --- | --- |
| **관측자가 게을러서 늦었다** | 무전 점유 시 긴급 보고 우선 규칙이 없어 보고가 110초 지연되었다 |
| **지휘자가 헷갈렸다** | 구역 할당표에 2~3시 중복이 있었고 확인 절차가 없었다 |

마. 작성 서식 2 - 개선점 목록(5건 이상) 및 SOP 1차 수정안

| **번호** | **개선점** | **SOP 현행** | **SOP 수정** | **근거(매트릭스 행)** |
| --- | --- | --- | --- | --- |
| **1** |  |  |  |  |
| **2** |  |  |  |  |
| **3** |  |  |  |  |
| **4** |  |  |  |  |
| **5** |  |  |  |  |
| **6** |  |  |  |  |

| **점검 항목** | **확인** |
| --- | --- |
| 4단계 순서로 AAR을 진행했다 | ( ) |
| 진술과 기록이 다른 곳을 모두 표시했다 | ( ) |
| 모든 원인 항목에 근거(F번호·기록)를 연결했다 | ( ) |
| 개선점이 5건 이상이다 | ( ) |
| 회의록에 개인 비난 표현이 없다 | ( ) |
| 개선점이 SOP 수정안에 반영되었다 | ( ) |

**



---

## 🎯 미션 09. 피드백 반영 역할별 보고 및 사격 절차 재훈련을 통한 반응 속도 개선

### 🤖 AI Co-pilot 협업 가이드 (프롬프트 질의 및 검증)
> **💡 프롬프트 질의 예시:**
> `"동일 조건 하 2차 재훈련 데이터를 1차와 일대일 대조하여 조치 시간 단축률(30% 목표)과 오류 감소율(50% 목표)을 계산하고 보수 교육 로드맵을 작성해줘."`
>
> **🔍 AI 결과 검증 포인트 (HITL):**
> - 성과 향상이 단순 암기 효과가 아닌 절차 숙달에 의한 것인지 동일 측정 기준으로 검증합니다.

---

### 📑 실습자료 및 과업 지침

피드백 반영 역할별 보고 및 사격 절차 재훈련을 통한 반응 속도 개선**

가. 제공 자료 - 재훈련 동일 조건 유지표

| **항목** | **1차(M4)** | **재훈련** | **동일 여부** |
| --- | --- | --- | --- |
| **주입 카드 구성** | 10장 | 같은 10장(순서만 변경) | ( ) |
| **역할 편성** |  |  | ( ) |
| **평가표** | M4 서식 다 | M4 서식 다 | ( ) |
| **측정 방법** | 초시계·시각 기록표 |  | ( ) |
| **시간대·환경** |  |  | ( ) |

나. 제공 자료 - 계산 예시 [교육용가상자료]

| 평균 조치 시간: 1차 210초 → 재훈련 140초
단축률 = (210 - 140) ÷ 210 × 100 = 33.3% → 목표 30% 이상 달성

오류 건수: 1차 12건 → 재훈련 5건
감소율 = (12 - 5) ÷ 12 × 100 = 58.3% → 목표 50% 이상 달성 |
| --- |

다. 작성 서식 1 - 전후 성과 비교표

| **지표** | **1차(M4)** | **재훈련** | **변화율(%)** | **목표** | **달성** |
| --- | --- | --- | --- | --- | --- |
| **평균 조치 시간(초)** |  |  |  | 30% 이상 단축 | ( ) |
| **평균 보고 지연(초)** |  |  |  | 참고 | ( ) |
| **SOP 준수율(%)** |  |  |  | 참고 | ( ) |
| **오류 건수(누락)** |  |  |  | - | - |
| **오류 건수(오보고)** |  |  |  | - | - |
| **오류 건수(절차 위반)** |  |  |  | - | - |
| **오류 합계** |  |  |  | 50% 이상 감소 | ( ) |

라. 작성 서식 2 - 미달 항목 원인 및 보수 교육 계획

| **미달 항목** | **원인** | **보완 방법** |
| --- | --- | --- |
| **미달 1** |  |  |
| **미달 2** |  |  |

| **보수 교육 주기** | ( )주마다 ( )분, 방법: ( ) |
| --- | --- |

| **숙련 유지 확인 지표** | ( ) ※ 예: 평균 보고 지연 60초 이내 유지 |
| --- | --- |

| **점검 항목** | **확인** |
| --- | --- |
| 1차와 같은 카드·평가표·측정 방법을 썼다 | ( ) |
| 조치 시간 단축률을 계산식대로 계산했다 | ( ) |
| 오류를 유형별로 세고 감소율을 계산했다 | ( ) |
| 미달 항목의 원인을 적었다 | ( ) |
| 보수 교육 주기를 제안했다 | ( ) |

**



In [ ]:
# [미션 9 실습 코드] 1차 vs 2차 모의훈련 측정 데이터 입력
# TODO: 1차 훈련과 2차 재훈련의 평균 시간과 오류 발생 건수를 입력하고 개선율을 산출하세요.
m4_data = {'avg_time': 185.0, 'errors': 8}  # 1차 성과
m9_data = {'avg_time': 0.0, 'errors': 0}    # TODO: 2차 측정값을 입력하세요
# compare_mission9_retrial(m4_data, m9_data)


---

## 🎯 미션 10. 드론 경보 시 관측자-지휘자-경계조 협업 표준운용절차(SOP) 완성

### 🤖 AI Co-pilot 협업 가이드 (프롬프트 질의 및 검증)
> **💡 프롬프트 질의 예시:**
> `"M1부터 M9까지의 실증 데이터를 종합하여 5개 장(개요-SOP-MSEL-루브릭-AAR) 훈련 패키지와 교관 지침서 최종본을 마크다운으로 편찬해줘."`
>
> **🔍 AI 결과 검증 포인트 (HITL):**
> - 타 부대 교관이 별도 교육 없이 즉시 훈련을 통제할 수 있는지, 장별 연계성에 모순이 없는지 최종 승인합니다.

---

### 📑 실습자료 및 과업 지침

드론 경보 시 관측자-지휘자-경계조 협업 표준운용절차(SOP) 완성**

가. 작성 서식 1 - 훈련 패키지 5개 장 구성표

| **장** | **포함 내용** | **반영할 산출물** | **완성(O/X)** |
| --- | --- | --- | --- |
| **1장 개요** | 훈련 목적, 대상, 소요 시간, 가상 상황 | M1 | ( ) |
| **2장 SOP** | 역할별 표준 행동, 흐름도, 요약 카드, 안전 조치 | M2·M8 수정안 | ( ) |
| **3장 시나리오** | MSEL, 오류 주입 카드, 스트레스 타임라인 | M3·M5·M7 | ( ) |
| **4장 평가 루브릭** | SOP 체크, 시각 기록표, 채점표 | M4·M6·M9 | ( ) |
| **5장 AAR** | AAR 진행 순서, 원인 분석 매트릭스, 개선 이력 | M8·M9 | ( ) |

나. 작성 서식 2 - 교관 지침서

| **준비물** | ( ) |
| --- | --- |

| **역할 배정** | ( ) |
| --- | --- |

| **시간표** | ( ) |
| --- | --- |

| **안전 통제** | ( ) |
| --- | --- |

| **채점 방법** | ( ) |
| --- | --- |

| **갱신 주기** | ( ) ※ 연간 교육훈련 계획과 연계 |
| --- | --- |

다. 작성 서식 3 - 성과 수치 일치 확인

| **교안에 적은 수치** | **교안 값** | **원본 기록(M4·M9)** | **일치(O/X)** |
| --- | --- | --- | --- |
| **1차 SOP 준수율** |  |  | ( ) |
| **1차 평균 보고 지연** |  |  | ( ) |
| **재훈련 조치 시간 단축률** |  |  | ( ) |
| **재훈련 오류 감소율** |  |  | ( ) |

라. 작성 서식 4 - 모의 검토 피드백 기록(다른 조 교관 역할)

| **번호** | **검토 의견** | **반영 여부** | **반영 내용** |
| --- | --- | --- | --- |
| **1** |  |  |  |
| **2** |  |  |  |
| **3** |  |  |  |
| **4** |  |  |  |
| **5** |  |  |  |

| **점검 항목** | **확인** |
| --- | --- |
| 5개 장이 모두 있다 | ( ) |
| 흐름도·요약 카드 등 시각 자료가 들어갔다 | ( ) |
| 교안 수치가 M4·M9 기록과 일치한다 | ( ) |
| 모의 검토 피드백 반영 내역을 기록했다 | ( ) |
| 실제 부대 정보·실제 음어가 없다 | ( ) |
| 사격은 모의·교전수칙 준수 문구가 있다 | ( ) |

**4. 평가 채점표**

미션마다 정량 기준은 PASS/FAIL로, 정성 기준은 상·중·하로 판정한다. 근거가 되는 기록지·서식 번호를 비고에 적는다.

| **미션** | **구분** | **정량 기준(PASS/FAIL)** | **정성 기준(상·중·하)** | **판정** |
| --- | --- | --- | --- | --- |
| **M1. 야전 부대 드론 경보 상황 분석 및 관측자-지휘자-경계조 역할 정의** | 필수 | 현장 체크 항목 15개 이상 정의, 7단계 전체에 대한 역할 분담 매트릭스 빈칸 0개(100% 매핑), 단계마다 책임자가 1명으로 지정되고 교관 체크리스트 대조에서 누락이 없는가. | 사례 카드의 문제점이 역할 정의에 반영되어 전술 상황이 현실적이고 소부대에 바로 적용 가능한가. | PASS / FAIL
상 / 중 / 하 |
| **M2. 관측자(식별/보고), 지휘자(전파/결심), 경계조(대피/사격) 간 표준 행동 지침서 작성** | 필수 | SOP 7단계 모두에 역할별 세부 절차가 100% 적혔고, 필수 안전/보안 조치 8개가 모두 포함(100%)되었으며, 교차 검토에서 지적된 모호 표현이 모두 수정되었는가. | 긴급 상황에서 한눈에 읽히도록 절차가 명확하고 간결하며, 긴급/일반 모드 구분이 분명한가. | PASS / FAIL
상 / 중 / 하 |
| **M3. 무전 약어 및 육안 식별 보고 템플릿(방위, 고도, 기종, 속도) 표준화** | 필수 | 보고 템플릿에 필수 7항목이 모두 있고, 주입 사건 10건 이상·정상/비정상 6:4 비율을 지켰으며, 워크스루에서 나온 개연성 문제가 모두 수정되었는가. | 상황 전개가 개연성 있고 훈련 내내 적절한 긴장도가 유지되도록 사건 간격과 난이도가 배치되었는가. | PASS / FAIL
상 / 중 / 하 |
| **M4. 가상 드론 경보 발령 시 역할별 초동 조치 및 보고 체계 1차 모의훈련(Dry-run)** | 선택 | SOP 단계 준수율 80% 이상, 최초 포착에서 지휘자 수신까지 보고 지연시간 3분 이내, 병목·혼선 구간 2건 이상 식별했는가. | 현장 판단이 정확하고 보고가 표준 템플릿 순서대로 이루어졌는가. | PASS / FAIL
상 / 중 / 하 |
| **M5. 잘못된 방위 보고 및 통신 지연 등 인위적 오류 주입 시 교차 확인 훈련** | 필수 | 주입된 기만·오경보 식별 성공률 85% 이상, 실제 위협을 오경보로 잘못 버린 건 0건, 판정마다 교차 확인 근거가 기록되었는가. | 의심 신호를 놓치지 않고, 교차 확인 근거가 관측 사실에 기반해 타당한가. | PASS / FAIL
상 / 중 / 하 |
| **M6. 대공 사격 구역 할당 및 인접 분대 상황 전파 인계 실습** | 선택 | 구역 중복·공백 0개, 인계 필수 항목 누락률 0%, 인계 소요 10분 이내, 인수자 퀴즈 5문항 중 4문항 이상 정답인가. | 인계 전후로 상황 인식이 끊기지 않고, 인수자의 질문에 명확히 답했는가. | PASS / FAIL
상 / 중 / 하 |
| **M7. 야간 기습 및 다수 드론 동시 출현 극한 상황 모의 역할분담 스트레스 훈련** | 필수 | 비상 절차 전환 시간 1분 이내, 위험 조치 성공률 90% 이상, 지휘 공백 시 1분 안에 위임이 이루어지고 안전 수칙 위반 0건인가. | 위기 상황에서 우선순위를 냉정하게 판단하고 역할 간 협업이 유지되었는가. | PASS / FAIL
상 / 중 / 하 |
| **M8. AAR 회의를 통한 보고 지연 및 사격 구역 중복 문제점 분석** | 선택 | 개선점 5건 이상, 원인 분석 항목마다 기록지 근거 100% 연결, 회의록에 개인 비난 표현 0건인가. | 비난 없이 사실 중심으로 토의했고 원인 규명이 객관적인가. | PASS / FAIL
상 / 중 / 하 |
| **M9. 피드백 반영 역할별 보고 및 사격 절차 재훈련을 통한 반응 속도 개선** | 필수 | 1차 대비 평균 조치 시간 30% 이상 단축, 오류 발생 건수 50% 이상 감소, 동일 평가표·동일 측정 방법을 사용했는가. | 행동 절차가 몸에 익어 역할 간 호흡이 맞고 말없이도 다음 행동이 이어지는가. | PASS / FAIL
상 / 중 / 하 |
| **M10. 드론 경보 시 관측자-지휘자-경계조 협업 표준운용절차(SOP) 완성** | 필수 | 5개 장 100% 완비, 교안에 적힌 성과 수치가 M4·M9 기록과 일치하고, 모의 검토 피드백 반영 내역이 기록되었는가. | 교재가 완성도 있고 시각 자료가 이해를 돕으며 다른 부대에 전파할 가치가 있는가. | PASS / FAIL
상 / 중 / 하 |

| 이수 기준(권장): 필수 미션 7개가 모두 PASS이고 정성 판정에 '하'가 2개 이하일 것. 선택 미션(M4·M6·M8)은 수행 시 가점으로 반영하며 세부 배점은 교관이 정한다. |
| --- |

| **평가관** | 소속·성명: ( ) 서명: ( ) 일자: ( ) |
| --- | --- |

| **종합 의견** | ( ) |
| --- | --- |



In [ ]:
# [미션 10 실습 코드] 최종 5개 장 훈련 패키지 마크다운 자동 Export
import os
os.makedirs('package', exist_ok=True)
package_path = 'package/ops08_standard_training_package.md'
with open(package_path, 'w', encoding='utf-8') as f:
    f.write('# [OPS08] 소부대 드론 경보 협업 표준운용절차(SOP) 및 표준 훈련 패키지\n')
    f.write('## 제1장 개요 | 제2장 SOP | 제3장 MSEL 시나리오 | 제4장 평가루브릭 | 제5장 AAR 가이드\n\n')
    f.write('본 문서는 M1~M9 훈련 실증 데이터를 집대성한 전군 야전부대 배포용 표준 패키지입니다.\n')
print(f'✔ 최종 훈련 패키지 파일 내보내기 완료: {package_path}')
